# Platform Narrative

This notebook is the canonical architecture story for the ML deployment reference.

## Executive summary

We are building a **notebook-first ML platform reference** with:

- **MLflow** as the system of record for experiments, runs, models, and lineage
- **PostgreSQL + S3** as the MLflow persistence baseline (MinIO locally for parity)
- **Lambda.ai + Slurm** for distributed training and inference coordination
- **AWS + Kubernetes** for non-Lambda platform services and production operations
- **python-terraform** for Python-managed infrastructure workflows
- **Web UI + immutable notebook revisions** for controlled execution without notebook editing

## Platform architecture

```{mermaid}
flowchart LR
    U["ML Engineer"] --> UI["Notebook Web UI (immutable revision selection)"]
    UI --> ORCH["Execution Orchestrator (contract and policy checks)"]

    subgraph LOCAL["Local parity environment"]
      LOC_RUN["Local runner"]
      LOC_PG["PostgreSQL"]
      LOC_S3["MinIO or S3-compatible storage"]
      LOC_K8S["Kubernetes-like local control plane"]
      LOC_SLRM["Slurm-like local scheduling path"]
    end

    subgraph LAMBDA["Lambda.ai distributed compute"]
      SLRM["Slurm controller"]
      TRAIN["Distributed training and inference jobs"]
    end

    subgraph AWS["AWS platform services"]
      K8S["EKS or Kubernetes services"]
      S3["S3 artifacts"]
      RDS["RDS PostgreSQL"]
    end

    ORCH --> LOC_RUN
    ORCH --> SLRM
    ORCH --> K8S

    LOC_RUN --> MLFLOW["MLflow tracking and registry"]
    TRAIN --> MLFLOW
    K8S --> MLFLOW

    MLFLOW --- LOC_PG
    MLFLOW --- LOC_S3
    MLFLOW --- RDS
    MLFLOW --- S3

    LOC_K8S -."local parity".-> K8S
    LOC_SLRM -."local parity".-> SLRM

    MLFLOW --> OBS["Observability and cost visibility"]
    OBS --> U
```


## Reading lenses

### Lifecycle lens
Data and lineage -> training and experimentation -> model registry and promotion -> serving and observability.

### Topology lens
Local parity -> Lambda.ai Slurm distributed compute -> AWS Kubernetes platform services.

### Operations lens
Terraform-driven environment provisioning, security and governance controls, and cost attribution.

## Non-negotiable constraints represented

1. Python-first, Linux-only, PyTorch + GPU posture
2. Docker-first reproducibility, with Nix as helper layer
3. MLflow with PostgreSQL backend and S3 artifact storage
4. Lambda.ai Slurm for distributed coordination and failure handling
5. AWS Kubernetes for non-Lambda platform services
6. python-terraform as default infrastructure workflow
7. Security, lineage, experiment/model traceability, and reproducibility as hard requirements

## Implementation steps

1. **Bootstrap local environment** with Nix + uv, then validate notebook export/docs/test loop.
2. **Define immutable notebook intake** and execution contracts through Web UI boundaries.
3. **Implement local parity services** (MLflow + PostgreSQL/S3-compatible artifacts) for end-to-end iteration.
4. **Wire execution backends** for local, Lambda.ai Slurm, and AWS Kubernetes targets.
5. **Harden traceability path** from notebook revision to run metadata, model version, deployment, and prediction logs.
6. **Automate infrastructure** with python-terraform modules and environment-specific configuration.
7. **Promote through controlled gates** using reproducibility, security, and observability checks.

## Trade-offs

- **Notebook-first speed vs governance overhead**: faster experimentation, but stricter intake and review controls are required.
- **Local parity realism vs maintenance cost**: strong confidence in promotion, but extra platform setup burden.
- **Multi-backend flexibility vs operational complexity**: portability across local/Slurm/K8s, with increased orchestration and debugging surface.
- **Rich traceability vs storage/latency overhead**: better auditability and reproducibility, with additional metadata capture cost.

## Security considerations

- Enforce immutable revision execution: no ad-hoc notebook mutation through execution surfaces.
- Keep secrets out of notebooks and artifacts; inject credentials at runtime via environment-specific secret stores.
- Require role-based access controls for trigger, promotion, and deployment actions.
- Preserve audit trails linking actor, revision, config, and resulting model/deployment records.
- Validate input/output schemas and contract checks before execution and promotion.

## Examples of use

1. **Engineer local iteration**: select immutable revision -> run locally -> inspect MLflow run -> package/promote candidate model.
2. **Distributed training run**: submit approved notebook revision to Lambda.ai Slurm -> track metrics/artifacts in MLflow -> evaluate promotion gate.
3. **Platform deployment validation**: deploy approved model to AWS Kubernetes service path -> monitor prediction logs, drift signals, and cost attribution.

## Software engineer learning path

1. Read this notebook for architecture context.
2. Follow [Vertical slice reference](vertical_slice.html) to understand end-to-end traceability records.
3. Study [Execution backends](execution_backends.html) and [Notebook intake validation](notebook_intake.html).
4. Use [Terraform bootstrap](terraform_bootstrap.html) for infrastructure workflow patterns.

## ML researcher learning path

1. Start with [Data loading and exploration](data.html) and [Model training](model_training.html).
2. Review experiment/run traceability expectations in [MLflow parity](mlflow_parity.html).
3. Understand packaging/promotion implications through [Vertical slice reference](vertical_slice.html).
4. Coordinate with platform engineers on reproducibility and execution constraints before scaling runs.